# 02 — ETL (Spark) Exploration

Purpose: run the `clean_transform.py` join step interactively, inspect
schema/row counts at each stage, and eyeball a few rows before trusting
the production `run_etl()` entrypoint. Uses the SAME functions the
production pipeline calls (`src.etl.clean_transform`) -- this notebook
never reimplements the logic, only calls it and inspects the output.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import yaml
from src.etl.spark_session import get_spark_session, s3a_path
from src.etl.clean_transform import (
    read_raw_prices, read_raw_news, clean_prices, clean_news,
    aggregate_news_daily, join_prices_news,
)

with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

spark = get_spark_session(
    app_name=config["spark"]["app_name"] + "-Notebook",
    master=config["spark"]["master"],
)
spark

:: loading settings :: url = jar:file:/home/hadoop/BigData/software/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/moksha/.ivy2/cache
The jars for the packages stored in: /home/moksha/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2101a4f2-b2bc-454e-a5b9-d45bc24652fc;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 602ms :: artifacts dl 45ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evi

## 1. Read + clean raw prices

In [2]:
bucket = config["s3"]["bucket"]

raw_prices = read_raw_prices(spark, bucket, config["s3"]["paths"]["raw_prices"])
raw_prices.printSchema()
raw_prices.show(5)

26/07/27 15:17:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
INFO:equirisk.etl.clean_transform:Read raw prices: 168727 rows                  


root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)
 |-- ticker: string (nullable = true)



+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+------+
|               Date|              Open|              High|               Low|             Close|  Volume|Dividends|Stock Splits|ticker|
+-------------------+------------------+------------------+------------------+------------------+--------+---------+------------+------+
|2021-07-26 00:00:00|121.34072477285186|132.88063034771056|117.58427400853382|131.02842712402344|23955102|      0.0|         0.0|BSE.NS|
|2021-07-27 00:00:00| 131.2157397434908|134.00965672396137|125.90883365979214|129.35311889648438|11767698|      0.0|         0.0|BSE.NS|
|2021-07-28 00:00:00| 129.7693192416371| 129.7693192416371|124.35836845211486|127.36040496826172| 5096664|      0.0|         0.0|BSE.NS|
|2021-07-29 00:00:00| 127.9899642320122|131.94411764254096|126.01288038172908| 129.0409393310547| 6235110|      0.0|         0.0|BSE.NS|
|2021-07-30 00:00:00| 129.3218668102761| 

In [3]:
prices = clean_prices(raw_prices)
print("Row count after cleaning:", prices.count())
prices.show(5)

Row count after cleaning: 168727


+------+----------+------------------+------------------+------------------+------------------+--------+
|ticker|      date|              open|              high|               low|             close|  volume|
+------+----------+------------------+------------------+------------------+------------------+--------+
|   BSE|2021-07-27| 131.2157397434908|134.00965672396137|125.90883365979214|129.35311889648438|11767698|
|   BSE|2021-08-10|126.48242852283141|127.40327073890015| 117.4857788502584|120.69281768798828| 5238837|
|   BSE|2021-08-11|120.66107049046983| 121.0738593405101|113.36848443097719|118.49128723144531| 7532532|
|   BSE|2021-08-13|119.60264789730539|122.24871768234125|119.60264789730539|120.62403106689453| 2054637|
|   BSE|2021-08-16|120.86217277171338|120.86217277171338|117.60221069133674|118.23726654052734| 1862145|
+------+----------+------------------+------------------+------------------+------------------+--------+
only showing top 5 rows



## 2. Read + clean raw news, aggregate to daily

In [4]:
raw_news = read_raw_news(spark, bucket, config["s3"]["paths"]["raw_news"])
raw_news.printSchema()
raw_news.show(5, truncate=60)

INFO:equirisk.etl.clean_transform:Read raw news: 8 article rows                 


root
 |-- article_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- published_at: string (nullable = true)
 |-- ticker: string (nullable = true)

+------------------------------------+------------------------------------------------------------+------------------------------------------------------------+---------------------------+------+
|                          article_id|                                                       title|                                                 description|               published_at|ticker|
+------------------------------------+------------------------------------------------------------+------------------------------------------------------------+---------------------------+------+
|d4bba1cb-fec6-4e16-8da3-1543acce6f48|Icon Advisers Inc Buys EOG Resources Inc, Berry Global Gr...|GuruFocus Article or News written by insider and the topi...|2021-08-04T20:38:13.000000Z|   BSE|
|a7b30

In [5]:
news = clean_news(raw_news)
news_daily = aggregate_news_daily(news)
print("Distinct ticker-days with news:", news_daily.count())
news_daily.show(5, truncate=60)

Distinct ticker-days with news: 8


+------+----------+------------------------------------------------------------+------------------------------------------------------------+-------------+
|ticker| news_date|                                                   headlines|                                                descriptions|article_count|
+------+----------+------------------------------------------------------------+------------------------------------------------------------+-------------+
|   ACC|2024-04-25|[KKR to acquire student housing portfolio for $1.64 billi...|[KKR to acquire student housing portfolio for $1.64 billion]|            1|
|   BSE|2021-08-04|[Icon Advisers Inc Buys EOG Resources Inc, Berry Global G...|[GuruFocus Article or News written by insider and the top...|            1|
|   BDL|2026-01-29| [Top Analyst Reports for Applied Materials, Linde & Abbott]|[AMAT rides a semiconductor rebound as systems, services ...|            1|
|   BDL|2025-07-14|[Top Research Reports for Eli Lilly, Netflix 

## 3. Join and inspect the base table

In [6]:
base_table = join_prices_news(prices, news_daily)
print("Joined row count:", base_table.count())
base_table.show(10, truncate=60)

Joined row count: 168727


+------+----------+------------------+------------------+------------------+------------------+--------+---------+------------+-------------+
|ticker|      date|              open|              high|               low|             close|  volume|headlines|descriptions|article_count|
+------+----------+------------------+------------------+------------------+------------------+--------+---------+------------+-------------+
|   BSE|2021-07-27| 131.2157397434908|134.00965672396137|125.90883365979214|129.35311889648438|11767698|     NULL|        NULL|            0|
|   BSE|2021-08-10|126.48242852283141|127.40327073890015| 117.4857788502584|120.69281768798828| 5238837|     NULL|        NULL|            0|
|   BSE|2021-08-11|120.66107049046983| 121.0738593405101|113.36848443097719|118.49128723144531| 7532532|     NULL|        NULL|            0|
|   BSE|2021-08-13|119.60264789730539|122.24871768234125|119.60264789730539|120.62403106689453| 2054637|     NULL|        NULL|            0|
|   BS

In [7]:
# Sanity checks worth eyeballing before trusting this in production:
print("Nulls per column:")
from pyspark.sql import functions as F
base_table.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in base_table.columns]).show()

print("Distinct tickers:", base_table.select("ticker").distinct().count())
print("Date range:", base_table.agg(F.min("date"), F.max("date")).collect())

Nulls per column:


+------+----+----+----+---+-----+------+---------+------------+-------------+
|ticker|date|open|high|low|close|volume|headlines|descriptions|article_count|
+------+----+----+----+---+-----+------+---------+------------+-------------+
|     0|   0|   0|   0|  0|    0|     0|   168720|      168720|            0|
+------+----+----+----+---+-----+------+---------+------------+-------------+



Distinct tickers: 150


Date range: [Row(min(date)=datetime.date(2021, 7, 26), max(date)=datetime.date(2026, 7, 24))]


In [8]:
# Convert a slice to pandas just to visually inspect in a familiar format
sample_pd = base_table.filter(base_table.ticker == base_table.select("ticker").first()[0]).toPandas()
sample_pd.tail(10)


,ticker,date,open,high,low,close,volume,headlines,descriptions,article_count
1229,BSE,2026-04-09,3141.723120,3277.066508,3129.355799,3248.840820,7362810,None,None,0
1230,BSE,2026-04-23,3498.782393,3498.782393,3443.128964,3453.900635,2373666,None,None,0
1231,BSE,2026-04-24,3466.766650,3472.850719,3403.234076,3436.945312,2270468,None,None,0
1232,BSE,2026-05-11,3880.776023,3934.634133,3835.096318,3907.705078,4662297,None,None,0
1233,BSE,2026-05-27,4425.341349,4435.115404,4224.969331,4237.236816,4332377,None,None,0
1234,BSE,2026-06-08,3802.482310,3946.602623,3774.855143,3906.508301,3952740,None,None,0
1235,BSE,2026-06-19,3980.513249,4068.282020,3965.153812,4009.636475,2001904,None,None,0
1236,BSE,2026-07-09,3752.114828,3841.878340,3750.718604,3795.699951,2319842,None,None,0
1237,BSE,2026-07-22,3664.000000,3683.100098,3612.000000,3632.199951,1588231,None,None,0
1238,BSE,2026-07-23,3613.500000,3648.000000,3572.000000,3579.000000,1699600,None,None,0


## 4. Write out (optional -- only if this looks correct)

The production `run_etl()` does this write for you; only run the cell
below if you're deliberately overwriting `processed/features/` from
this notebook instead of via the pipeline.


In [ ]:
# out_path = s3a_path(bucket, config["s3"]["paths"]["processed_features"])
# base_table.write.mode("overwrite").partitionBy("ticker").parquet(out_path)
# print("Wrote to", out_path)

In [9]:
spark.stop()